In [1]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip -q install transformers datasets scikit-learn torch pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC
import scipy.linalg

/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

In [5]:
def get_bert_embeddings(model, data_loader, device):
    model = model.eval()
    embeddings = []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state
            cls_embeddings = hidden_states[:, 0, :]
            embeddings.append(cls_embeddings.cpu().numpy())
    return np.vstack(embeddings)

def get_rowspace_projection(W):
    if W.ndim == 1:
        W = W.reshape(1, -1)
    
    basis = scipy.linalg.orth(W.T)
    P_row = basis @ basis.T
    return P_row

def inlp(X, Z, n_iterations):
    X_projected = X.copy()
    P_final = np.eye(X.shape[1])
    
    for i in range(n_iterations):
        clf = LinearSVC(dual='auto', max_iter=2000)
        clf.fit(X_projected, Z)
        W = clf.coef_
        
        P_row = get_rowspace_projection(W)
        P_null = np.eye(X.shape[1]) - P_row
        
        P_final = P_null @ P_final
        X_projected = X_projected @ P_null.T
        
    return P_final, X_projected

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)

In [7]:
df_sample = pd.read_csv('Jigsaw data processing/inlp_subset.csv')

texts = df_sample['comment_text'].values
z = df_sample['Z'].values

# Load precomputed embeddings
X = np.load('Jigsaw data processing/bert_embeddings.npz')['X']

In [8]:
X_train, X_test, z_train, z_test = train_test_split(
    X, z, test_size=0.3, random_state=42
)

In [9]:
gender_clf = LogisticRegression(max_iter=1000)
gender_clf.fit(X_train, z_train)
z_pred_orig = gender_clf.predict(X_test)
print(f"Original Accuracy (Subset Z): {accuracy_score(z_test, z_pred_orig):.4f}")

/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packag

Original Accuracy (Subset Z): 0.8254


/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [ ]:
P, X_train_inlp = inlp(X_train, z_train, n_iterations=50)
X_test_inlp = X_test @ P.T

# Sauvegarder la matrice de projection P
np.save('projection_matrix.npy', P)
print("Matrice de projection P sauvegardée dans 'projection_matrix.npy'")

/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:34: RuntimeWarning: divide by zero encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:34: RuntimeWarning: overflow encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:34: RuntimeWarning: invalid value encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:35: RuntimeWarning: divide by zero encountered in matmul
  X_projected = X_projected @ P_null.T
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:35: RuntimeWarning: overflow encountered in matmul
  X_projected = X_projected @ P_null.T
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:35: RuntimeWarning: invalid value encountered in matmul
  X_projected = X_projected @ P_n

In [11]:
gender_clf_inlp = LogisticRegression(max_iter=1000)
gender_clf_inlp.fit(X_train_inlp, z_train)
z_pred_inlp = gender_clf_inlp.predict(X_test_inlp)
print(f"INLP Accuracy (Subset Z): {accuracy_score(z_test, z_pred_inlp):.4f}")

INLP Accuracy (Subset Z): 0.5455


/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packag

In [16]:
import wandb
import os

# Intégration du débiaisage dans le fine-tuning sur LIAR
print("Chargement du dataset LIAR pour fine-tuning avec débiaisage...")

from datasets import load_dataset
import csv
col_names = [
    "id", "label_text", "statement", "subject", "speaker", 
    "job_title", "state_info", "party_affiliation", 
    "barely_true_counts", "false_counts", "half_true_counts", 
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

raw_datasets = load_dataset(
    "csv",
    data_files={
        "train": "../../Data/LIAR/train.tsv",
        "validation": "../../Data/LIAR/valid.tsv",
        "test": "../../Data/LIAR/test.tsv",
    },
    delimiter="\t",
    column_names=col_names,
    quoting=csv.QUOTE_NONE,
)

label_mapping = {
    'pants-fire': 0, 'false': 1, 'barely-true': 2, 
    'half-true': 3, 'mostly-true': 4, 'true': 5
}

def map_labels(example):
    return {'label': label_mapping[example['label_text']]}

raw_datasets = raw_datasets.map(map_labels)

my_secret_key = os.environ.get("WANDB")
wandb.login(key=my_secret_key)
wandb.init(project="StatApp")

statement_train, y_train = raw_datasets["train"]["statement"], raw_datasets["train"]["label"]
statement_val, y_val = raw_datasets["validation"]["statement"], raw_datasets["validation"]["label"]
statement_test, y_test = raw_datasets["test"]["statement"], raw_datasets["test"]["label"]

from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch.nn as nn

class StatementDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = StatementDataset(statement_train, y_train, tokenizer)
val_dataset = StatementDataset(statement_val, y_val, tokenizer)
test_dataset = StatementDataset(statement_test, y_test, tokenizer)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Modèle débiassé
class DebiasedBertForSequenceClassification(nn.Module):
    def __init__(self, model_name, config, projection_matrix):
        super(DebiasedBertForSequenceClassification, self).__init__()
        self.bert = BertForSequenceClassification.from_pretrained(model_name, config=config)
        self.projection = nn.Linear(projection_matrix.shape[0], projection_matrix.shape[1], bias=False)
        self.projection.weight.data = torch.tensor(projection_matrix.T, dtype=torch.float32)
        self.projection.weight.requires_grad = False

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        debiased_embeddings = self.projection(cls_embeddings)
        logits = self.bert.classifier(debiased_embeddings)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        
        return {'loss': loss, 'logits': logits}

num_labels = len(set(y_train))
from transformers import BertConfig
config = BertConfig.from_pretrained('bert-base-uncased', num_labels=num_labels)
config.hidden_dropout_prob = 0.3
config.attention_probs_dropout_prob = 0.3

model = DebiasedBertForSequenceClassification('bert-base-uncased', config, P)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Lancement du fine-tuning avec débiaisage...")
trainer.train()

results = trainer.evaluate(test_dataset)
print("Résultats sur le test set :", results)

Chargement du dataset LIAR pour fine-tuning avec débiaisage...


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/accuracy,▁▃▅▇▅▄▃▇▇█
eval/f1,▂▄▇█▆▃▁▇▅█
eval/loss,█▅█▄▄▄▃▂▂▁
eval/precision,▁▃▃▇▄█▅▂▄▃
eval/recall,▁▃▅▇▅▄▃▇▇█
eval/runtime,█▁▃▁▆▄▂▅▂▅
eval/samples_per_second,▁█▆█▃▅▇▄▇▄
eval/steps_per_second,▁█▆█▃▅▇▄▇▄
train/epoch,▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇██
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
+3,...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Lancement du fine-tuning avec débiaisage...


/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
50,1.749700,1.790516,0.187695,0.116701,0.209177,0.187695
100,1.742100,1.764583,0.207944,0.151901,0.206671,0.207944
150,1.783400,1.777828,0.217290,0.180272,0.187941,0.217290
200,1.756200,1.741062,0.232866,0.189532,0.248167,0.232866
250,1.754100,1.782445,0.206386,0.150289,0.182474,0.206386
300,1.731700,1.745173,0.224299,0.175693,0.209614,0.224299
350,1.750000,1.748187,0.207944,0.100508,0.146222,0.207944


/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/User

KeyboardInterrupt: 